# Week 4 (1.5B)
1. Solve rates: the base 1.5B samples 8 answers for every GSM8K train problem (two shards, one per GPU).
2. Training: `grpo.train` with TRAIN_ARGS; it trains on GPU 0 and generates with vLLM on GPU 1.
Set DO_SOLVE_RATES / TRAIN_ARGS with kaggle/week4/configure.py.

In [ ]:
import os, subprocess
REPO, SRC = "https://github.com/Vansh8010-boop/grpo-reasoning.git", "/tmp/grpo-reasoning"
if os.path.isdir(SRC):
    subprocess.run(["git", "-C", SRC, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO, SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q vllm peft "math-verify[antlr4_13_2]" 2>&1 | tail -1
# vLLM brings its own torch; Kaggle's torchaudio no longer matches it and transformers imports it.
# Kaggle's torchao is too old for peft, which imports it if present. Neither is used here.
!pip uninstall -y -q torchaudio torchao
!python -c "import vllm, torch, transformers, peft; print('vllm', vllm.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)"
!python -m pytest -q tests/ 2>&1 | tail -2

In [ ]:
DO_SOLVE_RATES = False
OUT = "/kaggle/working"
env = {**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1"}
if DO_SOLVE_RATES:
    os.makedirs(f"{OUT}/filter", exist_ok=True)
    procs = [subprocess.Popen(["python", "-m", "grpo.solve_rates", "--model", "Qwen/Qwen2.5-1.5B-Instruct",
                               "--shard", str(g), "--num-shards", "2", "--out", f"{OUT}/filter/solve.{g}.jsonl"],
                              env={**env, "CUDA_VISIBLE_DEVICES": str(g)}, stdout=open(f"{OUT}/solve-{g}.log", "w"),
                              stderr=subprocess.STDOUT) for g in (0, 1)]
    print("solve-rate exit codes", [p.wait() for p in procs])
    rows = [l for g in (0, 1) for l in open(f"{OUT}/filter/solve.{g}.jsonl")]
    open(f"{OUT}/filter/solve-rates-1.5b.jsonl", "w").writelines(rows)
    os.makedirs("results/filter", exist_ok=True)
    open("results/filter/solve-rates-1.5b.jsonl", "w").writelines(rows)
    import collections, json
    print(len(rows), "problems; solved k/8:", sorted(collections.Counter(json.loads(l)["n_correct"] for l in rows).items()))

In [ ]:
RESUME_RUN = "grpo-1.5b"
# Continue a run from an earlier session: the earlier checkpoint folder is attached as a private
# Kaggle dataset; copy it to where grpo.train looks, and training picks up from ckpt-latest.
if RESUME_RUN:
    import glob, shutil
    found = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/ckpt-latest/trainer_state.pt", recursive=True)]
    assert len(found) == 1, f"expected one ckpt-latest under /kaggle/input, found {found}"
    shutil.copytree(found[0], f"{OUT}/checkpoints/{RESUME_RUN}")
    print("copied", found[0], "->", f"{OUT}/checkpoints/{RESUME_RUN}", sorted(os.listdir(f"{OUT}/checkpoints/{RESUME_RUN}")))

In [ ]:
TRAIN_ARGS = ["--config", "configs/grpo-1.5b.yaml", "--set", "steps=500", "out_dir=/kaggle/working/checkpoints"]
if TRAIN_ARGS:
    path = f"{OUT}/train.log"
    code = subprocess.run(["python", "-m", "grpo.train", *TRAIN_ARGS], env=env, stdout=open(path, "w"), stderr=subprocess.STDOUT).returncode
    print(" ".join(TRAIN_ARGS), "-> exit code", code)
    print("".join(l for l in open(path) if l.startswith(("step", "resumed", "trainable", "non-finite", "train filter")) or "Error" in l)[-8000:])